# Objetivo

Desenvolvimento do Teleutils em ambiente de teste local
Teste das funções de normalização de números telefônicos

# Configuração do Ambiente

## Bibliotecas

In [ ]:
# 1. Módulos da Biblioteca Padrão
import re
from pathlib import Path

# 2. Módulos de Terceiros
import pandas as pd

In [ ]:
# Configuração do Pandas
# Exibir todas as colunas
pd.set_option("display.max_columns", None)

# Exibir todas as linhas
pd.set_option("display.max_rows", None)

# Evitar que o texto nas células seja cortado
pd.set_option("display.max_colwidth", None)

In [ ]:
from teleutils.preprocessing import normalize_number

## Arquivos de dados

### Dados de origem

In [ ]:
file_sup_100 = "/assets/data/cdr/arquivos_auxiliares/numeros/sup_100.csv"
columns = {
    "prestadora": "str",
    "numero_formatado": int,
    "numero_original": "str",  
    "quantidade": "int"
}
df_sup_100 = pd.read_csv(file_sup_100,sep=",",names=columns,dtype=columns,index_col=False)
df_sup_100 = df_sup_100[["numero_original","numero_formatado"]].drop_duplicates()

df_sup_100.head()

In [ ]:
df_sup_100.shape

# Análise

## Desenvolvimento

In [ ]:
#: Resolução 749, art. 15
_CN_PATTERN = r"""
    (?:
        1[1-9]|
        2[12478]|
        3[1-578]|
        4[1-9]|
        5[1345]|
        6[1-9]|
        7[134579]|
        8[1-9]|
        9[1-9]
    )
"""

In [ ]:
#: Resolução 749, art. 15
_CN_PATTERN = r"""
    (?:
        1[1-9]|
        2[12478]|
        3[1-578]|
        4[1-9]|
        5[1345]|
        6[1-9]|
        7[134579]|
        8[1-9]|
        9[1-9]
    )
"""

# Resolução 749, art. 11
_STFC_PATTERN = r"[2-6][0-9]{7}"

# Resolução 749, art. 12
_SMP_PATTERN = r"[789][0-9]{8}"

# Resolução 749, art. 18
_CNG_PATTERN = r"""
    (?:
        [3589]00[0-9]{7}|
        303[0-9]{7}
    )
"""

_3_DIGIT_SERVICES_PATTERN = r"""
    (?:
        10[02-6]|
        11[125-8]|
        12[1357-9]|
        13[02-68]|
        14[25-8]|
        15[0-9]|
        16[0-8]|
        18[0158]|
        19[0-9]|
        911
    )
"""

_4_DIGIT_SERVICES_PATTERN = r"""
    (?:
        105[0-35-9]|
        106[013467]|
        1358|
        1746
    )
"""

_103_SERVICES_PATTERN = r"""
    103(?:
        1[2-579]|
        2[13-9]|
        3[124-9]|
        4[1-3578]|
        5[1-468]|
        6[139]|
        8[149]|
        9[168]
    )
"""

_106_SERVICES_PATTERN = r"""
    106(?:
        1[0-35-8]|
        2[0145]|
        3[0137]|
        4[37-9]|
        5[0-35]|
        6[016]|
        7[137]|
        8[5-8]|
        9[1359]
    )
"""

_SERVICES_PATTERN = rf"""
    (?:
        {_3_DIGIT_SERVICES_PATTERN}|
        {_4_DIGIT_SERVICES_PATTERN}|
        {_103_SERVICES_PATTERN}|
        {_106_SERVICES_PATTERN}
    )
"""

_CC_PATTERN = r"90(?:90)?"

# group(1): CN + fixo/móvel, ou CNG, sempre com o CN quando houver
LARGE_NUMBERS_PATTERN = re.compile(
    rf"""
    (
        (?:{_CN_PATTERN}|{_CC_PATTERN})
        (?:
            {_STFC_PATTERN}|
            {_SMP_PATTERN}
        )|
        {_CNG_PATTERN}
    )$
    """,
    re.VERBOSE,
)

# group(1): fixo ou móvel (8 ou 9 dígitos), sem CN
MID_NUMBERS_PATTERN = re.compile(
    rf"""
    ^(
        {_STFC_PATTERN}|
        {_SMP_PATTERN}
    )
    $
    """,
    re.VERBOSE,
)

# group(1): apenas o número do serviço, sem o CN
UTILITY_NUMBERS_PATTERN = re.compile(
    rf"""
    ^
    (?:{_CN_PATTERN})?
    ({_SERVICES_PATTERN})
    $
    """,
    re.VERBOSE,
)

In [ ]:
# Resolução 749, art. 11
_STFC_PATTERN = r"[2-6][0-9]{7}"

# Resolução 749, art. 12
_SMP_PATTERN = r"[789][0-9]{8}"

# Resolução 749, art. 18
_CNG_PATTERN = r"""
    (?:
        [3589]00[0-9]{7}|
        303[0-9]{7}
    )
"""

In [ ]:
# Serviços de 3 dígitos (ex.: 190), sem CN; compõem _SERVICES_PATTERN.
_3_DIGIT_SERVICES_PATTERN = r"""
    (?:
        10[02-6]|
        11[125-8]|
        12[1357-9]|
        13[02-68]|
        14[25-8]|
        15[0-9]|
        16[0-8]|
        18[0158]|
        19[0-9]|
        911
    )
"""

In [ ]:
_4_DIGIT_SERVICES_PATTERN = r"""
    (?:
        105[0-35-9]|
        106[013467]|
        133[12]|
        1358|
        1746
    )
"""

In [ ]:
_103_SERVICES_PATTERN = r"""
    103(?:
        1[2-579]|
        2[13-9]|
        3[124-9]|
        4[1-3578]|
        5[1-468]|
        6[139]|
        8[149]|
        9[168]
    )
"""

In [ ]:
_106_SERVICES_PATTERN = r"""
    106(?:
        1[0-35-8]|
        2[0145]|
        3[0137]|
        4[37-9]|
        5[0-35]|
        6[016]|
        7[137]|
        8[5-8]|
        9[1359]
    )
"""

In [ ]:
s = "5586#1#9090981785314"
LARGE_NUMBERS_PATTERN.search(s).group(1)

In [ ]:
_SERVICES_PATTERN = rf"""
    (?:
        {_3_DIGIT_SERVICES_PATTERN}|
        {_4_DIGIT_SERVICES_PATTERN}|
        {_103_SERVICES_PATTERN}|
        {_106_SERVICES_PATTERN}
    )
"""

In [ ]:
assert LARGE_NUMBERS_PATTERN.search("2740096725").group(1) == "2740096725"
assert LARGE_NUMBERS_PATTERN.search("27988236365").group(1) == "27988236365"
assert LARGE_NUMBERS_PATTERN.search("8001235700").group(1) == "8001235700"

assert MID_NUMBERS_PATTERN.search("40096725").group(1) == "40096725"
assert MID_NUMBERS_PATTERN.search("988236365").group(1) == "988236365"
assert MID_NUMBERS_PATTERN.search("40096725").group(1) == "40096725"
assert MID_NUMBERS_PATTERN.search("988236365").group(1) == "988236365"

assert UTILITY_NUMBERS_PATTERN.search("100").group(1) == "100"
assert UTILITY_NUMBERS_PATTERN.search("27100").group(1) == "100"

In [ ]:
# group(1): CN + fixo/móvel, ou CNG, sempre com o CN quando houver.
# Não possui âncora ``^``: com ``search``, aceita qualquer prefixo antes do
# trecho final (ex.: ``55`` + CN + número), que é descartado do resultado.
LARGE_NUMBERS_PATTERN = re.compile(
    rf"""
    (
        {_CN_PATTERN}
        (?:
            {_STFC_PATTERN}|
            {_SMP_PATTERN}
        )|
        {_CNG_PATTERN}
    )$
    """,
    re.VERBOSE,
)

In [ ]:
# group(1): fixo ou móvel (8 ou 9 dígitos), sem CN
MID_NUMBERS_PATTERN = re.compile(
    rf"""\
    ^(
        {_STFC_PATTERN}|
        {_SMP_PATTERN}
    )
    $
    """,
    re.VERBOSE,
)

In [ ]:
# group(1): apenas o número do serviço, sem o CN
UTILITY_NUMBERS_PATTERN = re.compile(
    rf"""
    ^
    (?:{_CN_PATTERN})?
    ({_SERVICES_PATTERN})
    $
    """,
    re.VERBOSE,
)

In [ ]:
assert UTILITY_NUMBERS_PATTERN.search("*100") is None
assert UTILITY_NUMBERS_PATTERN.search("b100") is None
assert UTILITY_NUMBERS_PATTERN.search("111").group(1) == "111"
assert UTILITY_NUMBERS_PATTERN.search("31111").group(1) == "111"
assert UTILITY_NUMBERS_PATTERN.search("50111") is None

assert UTILITY_NUMBERS_PATTERN.search("1331").group(1) == "1331"
assert UTILITY_NUMBERS_PATTERN.search("0221331") is None
assert UTILITY_NUMBERS_PATTERN.search("221332").group(1) == "1332"

assert UTILITY_NUMBERS_PATTERN.search("*8486") is None

assert UTILITY_NUMBERS_PATTERN.search("2110621").group(1) == "10621"
assert UTILITY_NUMBERS_PATTERN.search("100######") is None
assert UTILITY_NUMBERS_PATTERN.search("5527#909040400100") is None

In [ ]:
# Remove marcadores de discagem apenas nos contextos especificados pelas
# alternativas; não realiza limpeza geral de pontuação ou espaços.
CLEAN_PATTERN = re.compile(
    r"""
        ^[*#](?=.{5})|    # "*" ou "#" inicial, só se a string tiver mais de 5 caracteres
        [*#]+$|           # um ou mais "*" ou "#" no final
        \#9090            # prefixo de chamada a cobrar (#9090) no meio do número
    """,
    re.VERBOSE | re.DOTALL,
)

In [ ]:
assert CLEAN_PATTERN.sub("", "*100") == "*100"
assert CLEAN_PATTERN.sub("", "*8486") == "*8486"
assert CLEAN_PATTERN.sub("", "*909027988887777") == "909027988887777"
assert CLEAN_PATTERN.sub("", "027123######") == "027123"
assert CLEAN_PATTERN.sub("", "5527#909040400123") == "552740400123"

In [ ]:
PREFFIX_PATTERN = re.compile(
    r"""
    ^(?:
        90(?:90)?|  # prefixo de chamada a cobrar
        00|         # prefixo de discagem internacional
        0           # prefixo de discagem nacional
    )
    """,
    re.VERBOSE,
)

In [ ]:
assert PREFFIX_PATTERN.sub("", "9090123") == "123"
assert PREFFIX_PATTERN.sub("", "90123") == "123"
assert PREFFIX_PATTERN.sub("", "0055123") == "55123"
assert PREFFIX_PATTERN.sub("", "027123") == "27123"
assert PREFFIX_PATTERN.sub("", "27123") == "27123"

## Teste

In [ ]:
resultado = [normalize_number(n) for n in df_sup_100["numero_original"]]
df_sup_100[["numero_normalizado", "valido"]] = pd.DataFrame(
    resultado, index=df_sup_100.index
)
df_sup_100.head()

In [ ]:
df_sup_100[df_sup_100["numero_normalizado"]=="100"]["valido"].sum()

In [ ]:
df_sup_100[df_sup_100["valido"]==True]